In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 61.8 MB/s eta 0:00:00


In [ ]:
pip install joblib

In [ ]:
import numpy as np
import pickle
from gensim.models import Word2Vec
import joblib

save_dir = "/content/drive/MyDrive/NLP_Project_Preprocessing"

X_train_pad = np.load(f"{save_dir}/X_train_pad.npy")
X_test_pad = np.load(f"{save_dir}/X_test_pad.npy")

y_train_stance = np.load(f"{save_dir}/y_train_stance.npy")
y_test_stance = np.load(f"{save_dir}/y_test_stance.npy")

embedding_matrix = np.load(f"{save_dir}/embedding_matrix.npy")

with open(f"{save_dir}/word_index.pkl", "rb") as f:
    word_index = pickle.load(f)

with open(f"{save_dir}/stance_encoder.pkl", "rb") as f:
    stance_encoder = joblib.load(f)


word2vec_model = Word2Vec.load(f"{save_dir}/word2vec.model")

vocab_size = len(word_index) + 1
embedding_dim = embedding_matrix.shape[1]
max_len = X_train_pad.shape[1]

print(f"Training samples : {X_train_pad.shape[0]:,}")
print(f"Test samples     : {X_test_pad.shape[0]:,}")
print(f"Vocabulary size  : {vocab_size:,}")
print(f"Embedding dim    : {embedding_dim}")
print(f"Sequence length  : {max_len}")
print(f"Classes          : {len(stance_encoder.classes_)}")
print("Labels:", stance_encoder.classes_)

Training samples : 1,166,475
Test samples     : 291,619
Vocabulary size  : 81,957
Embedding dim    : 100
Sequence length  : 30
Classes          : 3
Labels: ['Pro Russia' 'Pro Ukraine' 'Unsure']


In [ ]:
pip install keras-tcn

INFO: pip is looking at multiple versions of keras-tcn to determine which version is compatible with other requirements. This could take a while.


In [ ]:
from tcn import TCN

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import SpatialDropout1D
from tensorflow.keras.optimizers import Adam
from tcn import TCN

model = Sequential([
    Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=False
    ),

    SpatialDropout1D(0.2),

    TCN(
        nb_filters=64,
        kernel_size=3,
        dilations=[1,2,4,8],
        dropout_rate=0.2,
        return_sequences=False
    ),

    Dense(32, activation='relu'),

    Dropout(0.5),

    Dense(3, activation='softmax')
])

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

In [ ]:
model.compile(
    optimizer="adam",
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train_stance)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_stance
)

class_weights = dict(zip(classes, weights))
print(class_weights)

{np.int64(0): np.float64(19.825871915153986), np.int64(1): np.float64(4.280186695727794), np.int64(2): np.float64(0.3681985189674438)}


In [ ]:
history = model.fit(
    X_train_pad,
    y_train_stance,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stopping]
)

Epoch 1/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 82s 9ms/step - accuracy: 0.4719 - loss: 1.0427 - val_accuracy: 0.5285 - val_loss: 0.8803
Epoch 2/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 65s 7ms/step - accuracy: 0.4831 - loss: 0.9608 - val_accuracy: 0.5256 - val_loss: 0.8858
Epoch 3/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 47s 6ms/step - accuracy: 0.4928 - loss: 0.9375 - val_accuracy: 0.5180 - val_loss: 0.8955


In [ ]:
import numpy as np

y_pred_probs = model.predict(X_test_pad)

y_pred = np.argmax(y_pred_probs, axis=1)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_stance,
        y_pred,
        target_names=[
            "Pro Russia",
            "Pro Ukraine",
            "Unsure"
        ]
    )
)

              precision    recall  f1-score   support

  Pro Russia       0.03      0.71      0.06      4902
 Pro Ukraine       0.24      0.47      0.32     22711
      Unsure       0.97      0.53      0.69    264006

    accuracy                           0.53    291619
   macro avg       0.41      0.57      0.36    291619
weighted avg       0.90      0.53      0.65    291619



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_stance, y_pred)

print(cm)

[[  3473    604    825]
 [  8141  10672   3898]
 [ 90827  32704 140475]]


##model 2

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import SpatialDropout1D
from tensorflow.keras.optimizers import Adam
from tcn import TCN

model = Sequential([
    Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=True
    ),

    SpatialDropout1D(0.2),

    TCN(
        nb_filters=64,
        kernel_size=3,
        dilations=[1,2,4,8],
        dropout_rate=0.2,
        return_sequences=False
    ),

    Dense(32, activation='relu'),

    Dropout(0.5),

    Dense(3, activation='softmax')
])

In [ ]:
model.compile(
    optimizer="adam",
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train_pad,
    y_train_stance,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stopping]
)

Epoch 1/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 86s 9ms/step - accuracy: 0.4973 - loss: 1.0280 - val_accuracy: 0.5836 - val_loss: 0.7947
Epoch 2/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 63s 8ms/step - accuracy: 0.5076 - loss: 0.9135 - val_accuracy: 0.6019 - val_loss: 0.7961
Epoch 3/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 62s 8ms/step - accuracy: 0.5354 - loss: 0.8668 - val_accuracy: 0.5753 - val_loss: 0.8266


In [ ]:
import numpy as np

y_pred_probs = model.predict(X_test_pad)

y_pred = np.argmax(y_pred_probs, axis=1)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 26s 3ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_stance,
        y_pred,
        target_names=[
            "Pro Russia",
            "Pro Ukraine",
            "Unsure"
        ]
    )
)

              precision    recall  f1-score   support

  Pro Russia       0.04      0.71      0.07      4902
 Pro Ukraine       0.29      0.45      0.35     22711
      Unsure       0.96      0.59      0.73    264006

    accuracy                           0.58    291619
   macro avg       0.43      0.59      0.39    291619
weighted avg       0.89      0.58      0.69    291619



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_stance, y_pred)

print(cm)

[[  3493    399   1010]
 [  7199  10213   5299]
 [ 82947  24553 156506]]


##model 3

In [ ]:
class_weights = {
    0: 10,
    1: 3,
    2: 1
}

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import SpatialDropout1D
from tensorflow.keras.optimizers import Adam
from tcn import TCN

model = Sequential([
    Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=True
    ),

    SpatialDropout1D(0.2),

    TCN(
        nb_filters=64,
        kernel_size=3,
        dilations=[1,2,4,8],
        dropout_rate=0.2,
        return_sequences=False
    ),

    Dense(32, activation='relu'),

    Dropout(0.5),

    Dense(3, activation='softmax')
])

In [ ]:
model.compile(
    optimizer="adam",
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train_pad,
    y_train_stance,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stopping]
)

Epoch 1/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 91s 9ms/step - accuracy: 0.8983 - loss: 0.9809 - val_accuracy: 0.9057 - val_loss: 0.3881
Epoch 2/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 64s 8ms/step - accuracy: 0.8964 - loss: 0.8986 - val_accuracy: 0.9103 - val_loss: 0.3872
Epoch 3/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 63s 8ms/step - accuracy: 0.8951 - loss: 0.8580 - val_accuracy: 0.9022 - val_loss: 0.4156
Epoch 4/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 63s 8ms/step - accuracy: 0.8926 - loss: 0.8285 - val_accuracy: 0.9022 - val_loss: 0.3828
Epoch 5/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 62s 8ms/step - accuracy: 0.8915 - loss: 0.8062 - val_accuracy: 0.8931 - val_loss: 0.3867
Epoch 6/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 63s 8ms/step - accuracy: 0.8913 - loss: 0.7861 - val_accuracy: 0.8935 - val_loss: 0.3859


In [ ]:
import numpy as np

y_pred_probs = model.predict(X_test_pad)

y_pred = np.argmax(y_pred_probs, axis=1)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 27s 3ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_stance,
        y_pred,
        target_names=[
            "Pro Russia",
            "Pro Ukraine",
            "Unsure"
        ]
    )
)

              precision    recall  f1-score   support

  Pro Russia       0.16      0.13      0.14      4902
 Pro Ukraine       0.55      0.30      0.39     22711
      Unsure       0.93      0.97      0.95    264006

    accuracy                           0.90    291619
   macro avg       0.55      0.47      0.49    291619
weighted avg       0.89      0.90      0.89    291619



In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_stance, y_pred)

print(cm)

[[   631    157   4114]
 [   461   6857  15393]
 [  2856   5457 255693]]


##model 4

In [ ]:
class_weights = {
    0: 15,
    1: 4,
    2: 1
}

In [ ]:
model = Sequential([
    Embedding(
        input_dim=embedding_matrix.shape[0],
        output_dim=embedding_matrix.shape[1],
        weights=[embedding_matrix],
        trainable=True
    ),

    SpatialDropout1D(0.2),

    TCN(
        nb_filters=64,
        kernel_size=3,
        dilations=[1,2,4,8],
        dropout_rate=0.2,
        return_sequences=False
    ),

    Dense(32, activation='relu'),

    Dropout(0.5),

    Dense(3, activation='softmax')
])

In [ ]:
model.compile(
    optimizer="adam",
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train_pad,
    y_train_stance,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    class_weight=class_weights,
    callbacks=[early_stopping]
)

Epoch 1/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 93s 10ms/step - accuracy: 0.8867 - loss: 1.2442 - val_accuracy: 0.8980 - val_loss: 0.4508
Epoch 2/10
8202/8202 ━━━━━━━━━━━━━━━━━━━━ 64s 8ms/step - accuracy: 0.8773 - loss: 1.1345 - val_accuracy: 0.9006 - val_loss: 0.4368


In [ ]:
import numpy as np

y_pred_probs = model.predict(X_test_pad)

y_pred = np.argmax(y_pred_probs, axis=1)

9114/9114 ━━━━━━━━━━━━━━━━━━━━ 21s 2ms/step


In [ ]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test_stance,
        y_pred,
        target_names=[
            "Pro Russia",
            "Pro Ukraine",
            "Unsure"
        ]
    )
)

              precision    recall  f1-score   support

  Pro Russia       0.00      0.00      0.00      4902
 Pro Ukraine       0.43      0.35      0.38     22711
      Unsure       0.93      0.96      0.95    264006

    accuracy                           0.90    291619
   macro avg       0.45      0.44      0.44    291619
weighted avg       0.87      0.90      0.89    291619



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test_stance, y_pred)

print(cm)

[[     0    359   4543]
 [     0   7851  14860]
 [     0  10039 253967]]
